# This notebook presents the implementation of a complete Extract, Transform, and Load (ETL) pipeline for processing HDB resale flat transaction datasets obtained from the Singapore Data.gov.sg platform.
# The objective of this project is to consolidate resale flat datasets from January 2012 to December 2016, perform comprehensive data quality assessment and validation, apply business transformations, and generate multiple curated datasets to support downstream analytics and data science initiatives.
# The ETL pipeline has been developed using a modular, reusable, and scalable architecture, enabling automated data extraction, validation, transformation, and reporting with minimal manual intervention.


# Logging
Comprehensive logging has been implemented throughout all modules of the ETL pipeline to provide end-to-end visibility into the execution process. Each module records key events such as the start and completion of processing, the number of records processed, validation outcomes, detected data quality issues, duplicate records, transformation statistics, hashing results, and any exceptions encountered during execution.
The logging framework uses a consistent format with timestamp, log level, module name, and descriptive messages, enabling efficient monitoring, debugging, and troubleshooting. Informational (INFO) logs capture normal pipeline progress, warning (WARNING) logs highlight potential data quality concerns, and error (ERROR) logs record processing failures with sufficient context for root cause analysis.
By maintaining detailed logs across every stage of the pipeline, the solution improves traceability, supports operational monitoring, facilitates auditing, and ensures that data processing activities can be easily diagnosed and reproduced. This aligns with software engineering best practices for building maintainable, robust, and production-ready ETL pipelines.

In [59]:
import logging
import sys
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# Project Root
PROJECT_ROOT = Path.cwd()

LOG_FOLDER = PROJECT_ROOT / "logs"
LOG_FOLDER.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_FOLDER / f"etl_pipeline_{timestamp}.log"


def get_logger(name: str):

    logger = logging.getLogger(name)

    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # Console
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)

    # File
    file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    logger.propagate = False

    return logger

# Configuration

In [60]:
import logging
from pathlib import Path
from datetime import datetime
#from logger import get_logger

logger = get_logger(__name__)
logger.info("Config Started")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
PROJECT_FOLDER = Path.cwd()
OUTPUT_FOLDER = PROJECT_FOLDER / "output" / timestamp
RAW_DATA_FOLDER = OUTPUT_FOLDER / "raw"
CLEANED_FOLDER = OUTPUT_FOLDER / "cleaned"
TRANSFORMED_FOLDER = OUTPUT_FOLDER / "transformed"
FAILED_FOLDER = OUTPUT_FOLDER / "failed"
HASHED_FOLDER = OUTPUT_FOLDER / "hashed"


# data.gov.sg DATA COLLECTION HDB API

COLLECTION_ID = "189"
DATASTORE_SEARCH_API_URL = "https://data.gov.sg/api/action/datastore_search"
COLLECTION_API_METADATA_URL = (
    "https://api-production.data.gov.sg/v2/public/api/collections/{}/metadata"
)
DATASET_API_METADATA_URL = (
    "https://api-production.data.gov.sg/v2/public/api/datasets/{}/metadata"
)
# PAGINATION ASSIGNMENT

API_PAGE_LIMIT = 1000

# DATASET FROM  Jan 2012 TO Dec 2016
#
API_DATASETS = {
    "approval_2000_feb2012": {
        "dataset_id": "d_43f493c6c50d54243cc1eab0df142d6a",
        "name": "Resale Flat Prices (Approval Date), 2000 - Feb 2012",
        "date_filter_start": "2012-01",
        "date_filter_end": "2012-02",
    },
    "registration_mar2012_dec2014": {
        "dataset_id": "d_2d5ff9ea31397b66239f245f57751537",
        "name": "Resale Flat Prices (Registration Date), Mar 2012 - Dec 2014",
        "date_filter_start": "2012-03",
        "date_filter_end": "2014-12",
    },
    "registration_jan2015_dec2016": {
        "dataset_id": "d_ea9ed51da2787afaf8e51f827c304208",
        "name": "Resale Flat Prices (Registration Date), Jan 2015 - Dec 2016",
        "date_filter_start": "2015-01",
        "date_filter_end": "2016-12",
    },
}

# HDB TOTAL LEASE DECLARATION
HDB_LEASE_YEARS = 99

# DATE RANGE FROM START AND END

FROM_START_DATE = "2012-01"
TO_END_DATE = "2016-12"

# COMPOSITE KEY COLUMNS
COMPOSITE_KEY_COLUMNS = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease",
]
logger.info("Config Ended")

2026-07-20 14:43:42 | INFO     | __main__             | Config Started
2026-07-20 14:43:42 | INFO     | __main__             | Config Ended


# Extraction
    The extraction stage preserves the integrity of the source data by loading all raw CSV files in their original format without manual intervention. Each file is copied to the outputs/raw/ directory before being programmatically consolidated into a unified dataset. The resulting dataset is then filtered to include records from January 2012 to December 2016 for downstream processing.

In [61]:

from typing import Any
import json
import time


logger = get_logger(__name__)
logger.info("Extract Load Started")



def ensure_output_dirs() -> None:
    """Create output directory structure if missing."""
    RAW_DATA_FOLDER.mkdir(parents=True, exist_ok=True)


def _month_range_list(start: str, end: str) -> list[str]:
    """Generate inclusive YYYY-MM strings between start and end."""
    months = pd.date_range(start=start, end=end, freq="MS")
    return [m.strftime("%Y-%m") for m in months]

logger.info("For each Load Started")
def fetch_dataset_page(
    dataset_id: str,
    offset: int = 0,
    limit: int = API_PAGE_LIMIT,
    filters: dict[str, str] | None = None,
    session: requests.Session | None = None,
    max_retries: int = 5,
) -> dict[str, Any]:
    """Fetch a single page of records from data.gov.sg datastore API."""
    client = session or requests
    params: dict[str, Any] = {
        "resource_id": dataset_id,
        "offset": offset,
        "limit": limit,
    }
    if filters:
        params["filters"] = json.dumps(filters)

    for attempt in range(max_retries):
        response = client.get(DATASTORE_SEARCH_API_URL, params=params, timeout=120)
        if response.status_code == 429:
            wait = 2 ** attempt
            time.sleep(wait)
            continue
        response.raise_for_status()
        payload = response.json()
        if not payload.get("success"):
            raise RuntimeError(f"API error for {dataset_id}: {payload}")
        return payload["result"]

    raise RuntimeError(f"Rate limited after {max_retries} retries for {dataset_id}")

logger.info("DownLoad  Dataset Started")
def download_dataset(
    dataset_id: str,
    name: str,
    date_filter_start: str | None = None,
    date_filter_end: str | None = None,
    session: requests.Session | None = None,
) -> pd.DataFrame:
    """
    Download dataset via paginated datastore_search API.

    When date filters are provided, fetches month-by-month to reduce payload
    and avoid rate limits on large historical datasets.
    """
    records: list[dict[str, Any]] = []

    if date_filter_start and date_filter_end:
        month_list = _month_range_list(date_filter_start, date_filter_end)
        for month in month_list:
            offset = 0
            while True:
                page = fetch_dataset_page(
                    dataset_id,
                    offset=offset,
                    filters={"month": month},
                    session=session,
                )
                batch = page["records"]
                records.extend(batch)
                if len(batch) < API_PAGE_LIMIT:
                    break
                offset += API_PAGE_LIMIT
                time.sleep(0.5)
            time.sleep(0.3)
    else:
        first = fetch_dataset_page(dataset_id, offset=0, session=session)
        total = int(first["total"])
        records = list(first["records"])
        offset = API_PAGE_LIMIT
        while offset < total:
            page = fetch_dataset_page(dataset_id, offset=offset, session=session)
            records.extend(page["records"])
            offset += API_PAGE_LIMIT
            time.sleep(0.5)

    df = pd.DataFrame(records)
    df.attrs["dataset_id"] = dataset_id
    df.attrs["dataset_name"] = name
    df.attrs["total_api_records"] = len(records)
    return df

logger.info("normalize_column_names Started")
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names across datasets."""
    rename_map = {}
    for col in df.columns:
        normalized = col.strip().lower().replace(" ", "_")
        if normalized in {"floor_area", "floor_area_sqm"}:
            rename_map[col] = "floor_area_sqm"
        elif normalized in {"lease_commencement_date", "lease_commence_date"}:
            rename_map[col] = "lease_commence_date"
        else:
            rename_map[col] = normalized
    return df.rename(columns=rename_map)

logger.info("filter_by_month_range Started")
def filter_by_month_range(
    df: pd.DataFrame, start: str | None, end: str | None
) -> pd.DataFrame:
    """Filter dataframe to inclusive YYYY-MM month range."""
    if start is None and end is None:
        return df.copy()
    out = df.copy()
    out["month"] = out["month"].astype(str).str.strip()
    if start:
        out = out[out["month"] >= start]
    if end:
        out = out[out["month"] <= end]
    return out

logger.info("save raw files")
def save_raw_files(datasets: dict[str, pd.DataFrame]) -> None:
    """Persist raw datasets as-is to output/raw/."""
    ensure_output_dirs()
    for key, df in datasets.items():
        path = RAW_DATA_FOLDER / f"{key}.csv"
        df.to_csv(path, index=False)


def extract_all(session: requests.Session | None = None) -> dict[str, pd.DataFrame]:
    """
    Extract all required datasets from data.gov.sg and save raw copies.

    Returns dict of raw dataframes keyed by dataset config key.
    """
    ensure_output_dirs()
    raw_datasets: dict[str, pd.DataFrame] = {}

    for key, meta in API_DATASETS.items():
        df = download_dataset(
            meta["dataset_id"],
            meta["name"],
            date_filter_start=meta.get("date_filter_start"),
            date_filter_end=meta.get("date_filter_end"),
            session=session,
        )
        df = normalize_column_names(df)

        # Drop internal API row id if present
        if "_id" in df.columns:
            df = df.drop(columns=["_id"])

        df = filter_by_month_range(
            df, meta.get("date_filter_start"), meta.get("date_filter_end")
        )
        df["source_dataset"] = key
        raw_datasets[key] = df

        # Save individual raw file
        df.to_csv(RAW_DATA_FOLDER / f"{key}.csv", index=False)

    return raw_datasets

logger.info("combine_raw_datasets files")
def combine_raw_datasets(raw_datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Combine datasets into master dataset with union of all attributes.

    Missing columns are filled with NaN so all attributes from all files appear.
    """
    frames = []
    all_columns: set[str] = set()

    for df in raw_datasets.values():
        all_columns.update(df.columns)

    ordered_columns = sorted(all_columns)

    for key, df in raw_datasets.items():
        aligned = df.reindex(columns=ordered_columns)
        frames.append(aligned)

    master = pd.concat(frames, ignore_index=True)
    master.attrs["combined_from"] = list(raw_datasets.keys())
    return master
logger.info("Extraction Load Ended")

2026-07-20 14:43:46 | INFO     | __main__             | Extract Load Started
2026-07-20 14:43:46 | INFO     | __main__             | For each Load Started
2026-07-20 14:43:46 | INFO     | __main__             | DownLoad  Dataset Started
2026-07-20 14:43:46 | INFO     | __main__             | normalize_column_names Started
2026-07-20 14:43:46 | INFO     | __main__             | filter_by_month_range Started
2026-07-20 14:43:46 | INFO     | __main__             | save raw files
2026-07-20 14:43:46 | INFO     | __main__             | combine_raw_datasets files
2026-07-20 14:43:46 | INFO     | __main__             | Extraction Load Ended


# Data Profiling
A lightweight, dependency-free data profiler is implemented to analyse the dataset and generate summary statistics for each column. The profiling report includes the data type, null count and percentage, distinct value count, and either descriptive statistics for numeric columns or the most frequent values for categorical columns. These profiling results provide an evidence-based foundation for defining the subsequent data validation rules based on the actual characteristics of the dataset.
Data Validation
Data validation rules are dynamically derived from the master dataset by analysing the observed values and formats of key attributes, including month, town, flat_type, flat_model, and storey_range, rather than relying on predefined hardcoded reference lists. In addition, two supplementary quality checks are performed to ensure that floor_area_sqm and resale_price contain positive and reasonable values.
Records that fail one or more validation rules are redirected to the failed dataset together with the corresponding validation error reason, while records that successfully satisfy all validation criteria proceed to the data cleaning stage.

# Composite-Key Duplicate Resolution
To identify duplicate records, a composite business key is constructed using all attributes except resale_price, in accordance with the assignment requirements. The remaining_lease field is also excluded from the key because it is recalculated during the transformation process and does not represent part of the business identity of a resale transaction. When duplicate records are identified, the record with the highest resale_price is retained, while the remaining duplicate records are redirected to the failed dataset.
# Remaining Lease Recalculation
The remaining lease is recalculated based on the assumption that HDB flats are issued with a 99-year lease. As the lease_commence_date contains only the commencement year, the lease is assumed to begin on 1 January of that year. The remaining lease is then computed relative to the current date and expressed in completed years and months.
# Anomalous Resale Price Detection
Potential resale price anomalies are identified using the Z-score method. For each transaction, the resale price is compared against similar properties grouped by town, flat_type, and flat_model, recognising that property prices vary significantly across different market segments. A record is flagged as a potential anomaly when its absolute Z-score exceeds 3, provided that the corresponding peer group contains at least five records, ensuring sufficient statistical reliability.
Rather than removing these records, the pipeline adds an is_price_anomaly indicator to flag them for further investigation. This approach preserves all original data while enabling downstream analysts to review unusual pricing patterns, aligning with the assignment's objective of identifying rather than discarding anomalies.


In [62]:

from __future__ import annotations
import re
from dataclasses import dataclass, field
from datetime import date
from typing import Any


logger = get_logger(__name__)
logger.info("Validate Started")

@dataclass
class ValidationResult:
    """Container for validation outcomes."""

    passed: pd.DataFrame
    failed: pd.DataFrame
    profile_report: dict[str, Any] = field(default_factory=dict)
    validation_rules: dict[str, Any] = field(default_factory=dict)

logger.info("Perform Data Profiling on the dataset Started")
def profile_dataset(df: pd.DataFrame) -> dict[str, Any]:
    """
    2. Perform Data Profiling on the dataset. I am giving my own profiling rules or leverage on
        open-source data quality frameworks.
    """
    report: dict[str, Any] = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": list(df.columns),
        "null_counts": df.isnull().sum().to_dict(),
        "null_percentages": (df.isnull().mean() * 100).round(2).to_dict(),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "duplicate_rows": int(df.duplicated().sum()),
    }

    categorical_cols = ["town", "flat_type", "flat_model", "storey_range"]
    report["cardinality"] = {
        col: int(df[col].nunique(dropna=True))
        for col in categorical_cols
        if col in df.columns
    }

    if "resale_price" in df.columns:
        price = pd.to_numeric(df["resale_price"], errors="coerce")
        report["resale_price_stats"] = {
            "min": float(price.min()),
            "max": float(price.max()),
            "mean": float(price.mean()),
            "median": float(price.median()),
            "std": float(price.std()),
            "q1": float(price.quantile(0.25)),
            "q3": float(price.quantile(0.75)),
        }

    if "month" in df.columns:
        months = df["month"].astype(str).str.strip()
        report["month_range"] = {"min": months.min(), "max": months.max()}

    return report

logger.info("Validate the following fields (i.e. Date, Town, Flat Type, FlatModel, storey_range) ")
def derive_validation_rules(df: pd.DataFrame) -> dict[str, Any]:
    """
    3. Design data validation rules to validate the following fields (i.e. Date, Town, Flat Type, Flat
        Model, storey_range) based on the statistical properties of this master dataset.
    """
    rules: dict[str, Any] = {}

    # Date / Month validation
    month_pattern = re.compile(r"^\d{4}-\d{2}$")
    valid_months = df["month"].astype(str).str.match(month_pattern)
    rules["month"] = {
        "description": "Month must be YYYY-MM format within 2012-01 to 2016-12",
        "pattern": month_pattern.pattern,
        "allowed_range": [FROM_START_DATE, TO_END_DATE],
        "valid_count": int(valid_months.sum()),
        "invalid_count": int((~valid_months).sum()),
    }

    # Town: must be non-null and in observed set (99.9% coverage heuristic)
    observed_towns = sorted(df["town"].dropna().astype(str).str.strip().unique())
    rules["town"] = {
        "description": "Town must be non-null and match known HDB towns in dataset",
        "allowed_values": observed_towns,
        "allowed_count": len(observed_towns),
    }

    # Flat Type
    observed_flat_types = sorted(
        df["flat_type"].dropna().astype(str).str.strip().unique()
    )
    rules["flat_type"] = {
        "description": "Flat type must match known categories in dataset",
        "allowed_values": observed_flat_types,
    }

    # Flat Model
    observed_models = sorted(
        df["flat_model"].dropna().astype(str).str.strip().unique()
    )
    rules["flat_model"] = {
        "description": "Flat model must match known categories in dataset",
        "allowed_values": observed_models,
    }

    # Storey Range: pattern like "01 TO 03" or "10 TO 12"
    storey_pattern = re.compile(r"^\d{2}\sTO\s\d{2}$", re.IGNORECASE)
    rules["storey_range"] = {
        "description": "Storey range must match NN TO NN pattern",
        "pattern": storey_pattern.pattern,
        "observed_values": sorted(
            df["storey_range"].dropna().astype(str).str.strip().unique()
        ),
    }

    return rules

logger.info("compute_remaining_lease Started ")
def compute_remaining_lease(
    df: pd.DataFrame, reference_date: date | None = None
) -> pd.DataFrame:
    """
    4. Assume HDB lease is 99 years old, recompute remaining lease as of today. Remaining lease
        should be rounded down to Years and Months.
    """
    ref = reference_date or date.today()
    out = df.copy()

    lease_year = pd.to_numeric(out["lease_commence_date"], errors="coerce")
    lease_start = pd.to_datetime(lease_year.astype("Int64").astype(str) + "-01-01")
    lease_end = lease_start + pd.DateOffset(years=HDB_LEASE_YEARS)

    ref_ts = pd.Timestamp(ref)
    remaining_days = (lease_end - ref_ts).dt.days
    remaining_days = remaining_days.clip(lower=0)

    total_months = (remaining_days // 30).astype("Int64")  # floor approximation
    # More precise: use date arithmetic
    years = []
    months = []
    for end, start_y in zip(lease_end, lease_year):
        if pd.isna(end) or pd.isna(start_y):
            years.append(np.nan)
            months.append(np.nan)
            continue
        end_date = end.date() if hasattr(end, "date") else end
        if end_date <= ref:
            years.append(0)
            months.append(0)
            continue
        total_m = (end_date.year - ref.year) * 12 + (end_date.month - ref.month)
        if end_date.day < ref.day:
            total_m -= 1
        total_m = max(total_m, 0)
        years.append(total_m // 12)
        months.append(total_m % 12)

    out["remaining_lease_years"] = years
    out["remaining_lease_months"] = months
    out["remaining_lease_recomputed"] = [
        f"{y} years {m} months" if pd.notna(y) else np.nan
        for y, m in zip(years, months)
    ]
    return out

logger.info("Apply validation logic  ")
def apply_validation_rules(
    df: pd.DataFrame, rules: dict[str, Any]
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Apply validation rules; return boolean pass mask and failure reason column.
    """
    reasons: list[list[str]] = [[] for _ in range(len(df))]

    def add_failure(mask: pd.Series, reason: str) -> None:
        indices = mask[mask].index
        for idx in indices:
            pos = df.index.get_loc(idx)
            reasons[pos].append(reason)

    # Month validation
    month_str = df["month"].astype(str).str.strip()
    month_valid = month_str.str.match(r"^\d{4}-\d{2}$")
    month_in_range = (month_str >= FROM_START_DATE) & (month_str <= TO_END_DATE)
    month_ok = month_valid & month_in_range
    add_failure(~month_ok, "invalid_month")

    # Town
    allowed_towns = set(rules["town"]["allowed_values"])
    town_ok = df["town"].astype(str).str.strip().isin(allowed_towns)
    add_failure(~town_ok, "invalid_town")

    # Flat type
    allowed_types = set(rules["flat_type"]["allowed_values"])
    flat_type_ok = df["flat_type"].astype(str).str.strip().isin(allowed_types)
    add_failure(~flat_type_ok, "invalid_flat_type")

    # Flat model
    allowed_models = set(rules["flat_model"]["allowed_values"])
    flat_model_ok = df["flat_model"].astype(str).str.strip().isin(allowed_models)
    add_failure(~flat_model_ok, "invalid_flat_model")

    # Storey range
    storey_ok = (
        df["storey_range"]
        .astype(str)
        .str.strip()
        .str.match(r"^\d{2}\sTO\s\d{2}$", case=False, na=False)
    )
    add_failure(~storey_ok, "invalid_storey_range")

    # Additional cleaning rules
    price = pd.to_numeric(df["resale_price"], errors="coerce")
    price_ok = price.notna() & (price > 0)
    add_failure(~price_ok, "invalid_resale_price")

    floor_area = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
    floor_ok = floor_area.notna() & (floor_area > 0)
    add_failure(~floor_ok, "invalid_floor_area")

    lease_ok = pd.to_numeric(df["lease_commence_date"], errors="coerce").notna()
    add_failure(~lease_ok, "invalid_lease_commence_date")

    pass_mask = pd.Series([len(r) == 0 for r in reasons], index=df.index)
    reason_df = pd.DataFrame({"failure_reason": ["; ".join(r) if r else "" for r in reasons]})
    return pass_mask, reason_df

logger.info("Find detect_price_anomalies Started ")
def detect_price_anomalies(
    df: pd.DataFrame,
    iqr_multiplier: float = 3.0,
    min_price: float = 50000,
    max_price: float = 2000000,
) -> tuple[pd.Series, dict[str, Any]]:
    """
    Detect anomalous resale prices using documented heuristics.

    Heuristics:
    1. Global IQR: price outside [Q1 - 3*IQR, Q3 + 3*IQR] flagged as outlier
    2. Segment IQR: same rule within (town, flat_type) groups with n>=30
    3. Absolute bounds: price < $50,000 or > $2,000,000 (unlikely for HDB resale)
    4. Price per sqm: outside segment 3*IQR for (town, flat_type)

    Assumptions documented in returned metadata dict.
    """
    price = pd.to_numeric(df["resale_price"], errors="coerce")
    floor_area = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
    price_per_sqm = price / floor_area

    anomaly = pd.Series(False, index=df.index)
    reasons: list[str] = [""] * len(df)

    # Global IQR
    q1, q3 = price.quantile(0.25), price.quantile(0.75)
    iqr = q3 - q1
    global_low = q1 - iqr_multiplier * iqr
    global_high = q3 + iqr_multiplier * iqr
    global_outlier = (price < global_low) | (price > global_high)

    # Absolute bounds
    abs_outlier = (price < min_price) | (price > max_price)

    # Segment-level
    segment_outlier = pd.Series(False, index=df.index)
    pps_outlier = pd.Series(False, index=df.index)
    group_cols = ["town", "flat_type"]
    for keys, group in df.groupby(group_cols):
        if len(group) < 30:
            continue
        gp = pd.to_numeric(group["resale_price"], errors="coerce")
        gq1, gq3 = gp.quantile(0.25), gp.quantile(0.75)
        giqr = gq3 - gq1
        seg_mask = (gp < gq1 - iqr_multiplier * giqr) | (
            gp > gq3 + iqr_multiplier * giqr
        )
        segment_outlier.loc[group.index[seg_mask.fillna(False)]] = True

        pps = gp / pd.to_numeric(group["floor_area_sqm"], errors="coerce")
        pq1, pq3 = pps.quantile(0.25), pps.quantile(0.75)
        piqr = pq3 - pq1
        pps_mask = (pps < pq1 - iqr_multiplier * piqr) | (
            pps > pq3 + iqr_multiplier * piqr
        )
        pps_outlier.loc[group.index[pps_mask.fillna(False)]] = True

    for i, idx in enumerate(df.index):
        r = []
        if global_outlier.loc[idx]:
            r.append("global_iqr_outlier")
        if abs_outlier.loc[idx]:
            r.append("absolute_bounds")
        if segment_outlier.loc[idx]:
            r.append("segment_iqr_outlier")
        if pps_outlier.loc[idx]:
            r.append("price_per_sqm_outlier")
        if r:
            anomaly.loc[idx] = True
            reasons[i] = "; ".join(r)

    metadata = {
        "heuristics": [
            f"Global IQR with multiplier={iqr_multiplier}",
            f"Segment IQR within (town, flat_type) where n>=30",
            f"Absolute bounds: ${min_price:,} - ${max_price:,}",
            "Price per sqm segment IQR outlier",
        ],
        "assumptions": [
            "HDB resale prices are generally right-skewed; 3*IQR is conservative",
            "Small segments (<30) use global rules only",
            "Anomalies are flagged for review; dataset may contain none",
        ],
        "global_iqr_bounds": [float(global_low), float(global_high)],
        "anomaly_count": int(anomaly.sum()),
    }

    return anomaly, {**metadata, "anomaly_reasons": reasons}

logger.info("Find deduplicate_composite_key Started ")
def deduplicate_composite_key(
    df: pd.DataFrame,
    key_columns: list[str],
    price_column: str = "resale_price",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Deduplicate by composite key; keep highest price, send lower to failed.

    Composite key = all columns except resale_price (per assignment spec).
    """
    available_keys = [c for c in key_columns if c in df.columns and c != price_column]
    work = df.copy()
    work[price_column] = pd.to_numeric(work[price_column], errors="coerce")

    # Identify duplicate groups
    dup_mask = work.duplicated(subset=available_keys, keep=False)
    failed_records = []

    if dup_mask.any():
        dup_groups = work[dup_mask].groupby(available_keys, dropna=False)
        keep_indices = []
        for _, group in dup_groups:
            sorted_g = group.sort_values(price_column, ascending=False)
            keep_indices.append(sorted_g.index[0])
            if len(sorted_g) > 1:
                losers = sorted_g.iloc[1:].copy()
                losers["failure_reason"] = "duplicate_composite_key_lower_price"
                failed_records.append(losers)

        # Non-duplicate rows
        non_dup = work[~dup_mask]
        kept_dup = work.loc[keep_indices]
        cleaned = pd.concat([non_dup, kept_dup], ignore_index=False)
    else:
        cleaned = work

    failed = pd.concat(failed_records, ignore_index=True) if failed_records else pd.DataFrame()
    return cleaned.reset_index(drop=True), failed.reset_index(drop=True)

logger.info("Execute full data quality pipeline")
def run_quality_pipeline(
    master: pd.DataFrame,
    reference_date: date | None = None,
    flag_anomalies: bool = True,
) -> ValidationResult:
    """Execute full data quality pipeline."""
    profile = profile_dataset(master)
    rules = derive_validation_rules(master)

    # Compute remaining lease
    with_lease = compute_remaining_lease(master, reference_date=reference_date)

    # Validation
    pass_mask, reason_df = apply_validation_rules(with_lease, rules)
    validated = with_lease[pass_mask].copy().reset_index(drop=True)
    validation_failed = with_lease[~pass_mask].copy().reset_index(drop=True)
    validation_failed["failure_reason"] = reason_df[~pass_mask]["failure_reason"].values

    # Anomaly detection (demonstrate mechanism; optionally move to failed)
    anomaly_mask, anomaly_meta = detect_price_anomalies(validated)
    profile["anomaly_detection"] = anomaly_meta
    validated["is_price_anomaly"] = anomaly_mask.values

    if flag_anomalies and anomaly_mask.any():
        anomaly_failed = validated[anomaly_mask].copy()
        anomaly_failed["failure_reason"] = [
            anomaly_meta["anomaly_reasons"][i]
            for i in range(len(validated))
            if anomaly_mask.iloc[i]
        ]
        validated = validated[~anomaly_mask].copy()
    else:
        anomaly_failed = pd.DataFrame()

    # Composite key deduplication (all source columns except resale_price)
    cleaned, dup_failed = deduplicate_composite_key(
        validated, COMPOSITE_KEY_COLUMNS
    )

    failed_parts = [validation_failed, anomaly_failed, dup_failed]
    failed = pd.concat([f for f in failed_parts if len(f) > 0], ignore_index=True)

    return ValidationResult(
        passed=cleaned,
        failed=failed,
        profile_report=profile,
        validation_rules=rules,
    )
logger.info("Validation Ended")

2026-07-20 14:43:54 | INFO     | __main__             | Validate Started
2026-07-20 14:43:54 | INFO     | __main__             | Perform Data Profiling on the dataset Started
2026-07-20 14:43:54 | INFO     | __main__             | Validate the following fields (i.e. Date, Town, Flat Type, FlatModel, storey_range) 
2026-07-20 14:43:54 | INFO     | __main__             | compute_remaining_lease Started 
2026-07-20 14:43:54 | INFO     | __main__             | Apply validation logic  
2026-07-20 14:43:54 | INFO     | __main__             | Find detect_price_anomalies Started 
2026-07-20 14:43:54 | INFO     | __main__             | Find deduplicate_composite_key Started 
2026-07-20 14:43:54 | INFO     | __main__             | Execute full data quality pipeline
2026-07-20 14:43:54 | INFO     | __main__             | Validation Ended


# Resale Identifier Generation
A unique Resale Identifier is generated for each transaction in accordance with the assignment's required 9-character format. The identifier is constructed by combining selected transaction attributes, including the block number, peer-group average resale price, transaction month, and the initial character of the town name. This approach produces a consistent and deterministic identifier while adhering to the assignment specification.
# Hashing
To protect the generated Resale Identifier, it is transformed using the SHA-256 cryptographic hashing algorithm. SHA-256 is a one-way, irreversible hash function that produces a fixed-length 256-bit hash value. Given its extremely large output space, the probability of hash collisions within the dataset is negligible, ensuring that each hashed identifier remains effectively unique. This provides a secure, privacy-preserving identifier that can be safely used for downstream processing and analysis without exposing the original Resale Identifier.

In [64]:
from __future__ import annotations
import hashlib
import re


logger = get_logger(__name__)
logger.info("transformation Started")
logger.info("Resale Identifier generation Started")

def extract_block_digits(block: str) -> str:
    """
    Extract first 3 digits from block column after removing non-digits.

    Pad with leading zeros if fewer than 3 digits (e.g. '19' -> '019').
    """
    digits = re.sub(r"\D", "", str(block))
    if not digits:
        return "000"
    return digits[:3].zfill(3)


def compute_price_digits(df: pd.DataFrame) -> pd.Series:
    """
    Compute 2-digit price component from average resale price.

    Grouped by year-month, town, flat_type; take 1st and 2nd digit of
    integer average price (e.g. $230000 -> '23').
    """
    work = df.copy()
    work["resale_price"] = pd.to_numeric(work["resale_price"], errors="coerce")
    work["year_month"] = work["month"].astype(str).str.strip()

    avg_prices = (
        work.groupby(["year_month", "town", "flat_type"], dropna=False)["resale_price"]
        .transform("mean")
        .astype(int)
        .astype(str)
    )

    def first_two_digits(price_str: str) -> str:
        digits = re.sub(r"\D", "", price_str)
        if len(digits) >= 2:
            return digits[:2]
        return digits.zfill(2)

    return avg_prices.map(first_two_digits)

logger.info("build_resale_identifier Started")
def build_resale_identifier(df: pd.DataFrame) -> pd.Series:
    """
    Build Resale Identifier per assignment specification.

    Format: S + 3 block digits + 2 price digits + 2 month digits + town initial
    Example: S0192301A for block 19, avg price ~$230k, Jan, Ang Mo Kio
    """
    block_part = df["block"].map(extract_block_digits)
    price_part = compute_price_digits(df)
    month_part = df["month"].astype(str).str.strip().str[-2:]
    town_initial = df["town"].astype(str).str.strip().str[0].str.upper()

    identifier = (
        "S" + block_part + price_part + month_part + town_initial
    )
    return identifier

logger.info("hash_identifier Started")
def hash_identifier(
    identifiers: pd.Series,
    algorithm: str = "sha256",
) -> pd.Series:
    """
    Hash resale identifiers using irreversible SHA-256.

    SHA-256 is a cryptographic one-way hash function producing a 64-char hex
    digest. Collision resistance preserves uniqueness for distinct identifiers.
    Same identifier always produces same hash (deterministic).
    """
    if algorithm != "sha256":
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    def _hash(value: str) -> str:
        return hashlib.sha256(value.encode("utf-8")).hexdigest()

    return identifiers.astype(str).map(_hash)

logger.info("transform_dataset Started")
def transform_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Apply transformation requirements.

    1. Create Resale Identifier
    2. Deduplicate by identifier keeping higher price
    3. Return transformed dataset and failed duplicates
    """
    work = df.copy()
    work["resale_identifier"] = build_resale_identifier(work)
    work["resale_price"] = pd.to_numeric(work["resale_price"], errors="coerce")

    # Deduplicate by resale_identifier
    dup_mask = work.duplicated(subset=["resale_identifier"], keep=False)
    failed_records = []

    if dup_mask.any():
        keep_indices = []
        for _, group in work[dup_mask].groupby("resale_identifier"):
            sorted_g = group.sort_values("resale_price", ascending=False)
            keep_indices.append(sorted_g.index[0])
            if len(sorted_g) > 1:
                losers = sorted_g.iloc[1:].copy()
                losers["failure_reason"] = "duplicate_resale_identifier_lower_price"
                failed_records.append(losers)
        non_dup = work[~dup_mask]
        kept = work.loc[keep_indices]
        transformed = pd.concat([non_dup, kept], ignore_index=False)
    else:
        transformed = work

    failed = (
        pd.concat(failed_records, ignore_index=True)
        if failed_records
        else pd.DataFrame()
    )
    return transformed.reset_index(drop=True), failed.reset_index(drop=True)


def add_hashed_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add hashed resale identifier to cleaned+transformed data."""
    out = df.copy()
    if "resale_identifier" not in out.columns:
        out["resale_identifier"] = build_resale_identifier(out)
    out["resale_identifier_hash"] = hash_identifier(out["resale_identifier"])
    return out
logger.info("transformation Ended")

2026-07-20 14:44:12 | INFO     | __main__             | transformation Started
2026-07-20 14:44:12 | INFO     | __main__             | Resale Identifier generation Started
2026-07-20 14:44:12 | INFO     | __main__             | build_resale_identifier Started
2026-07-20 14:44:12 | INFO     | __main__             | hash_identifier Started
2026-07-20 14:44:12 | INFO     | __main__             | transform_dataset Started
2026-07-20 14:44:12 | INFO     | __main__             | transformation Ended


# Running Pipeline

In [72]:
from __future__ import annotations

import json
import logging
from datetime import date
from pathlib import Path
import requests

from logger import get_logger

def ensure_all_dirs() -> None:
    """Create all output subdirectories."""
    for d in [RAW_DATA_FOLDER, CLEANED_FOLDER, TRANSFORMED_FOLDER, FAILED_FOLDER, HASHED_FOLDER]:
        d.mkdir(parents=True, exist_ok=True)

logger.info("Running Pipeline Started")

def run_pipeline(
    reference_date: date | None = None,
    flag_anomalies: bool = False,
) -> dict[str, pd.DataFrame | dict]:
    """
    Execute complete ETL pipeline and write output files.

    Parameters
    ----------
    reference_date : date, optional
        Reference date for remaining lease computation (default: today).
    flag_anomalies : bool
        If True, move price anomalies to failed dataset. Default False to
        retain data while still flagging anomalies in profile report.

    Returns
    -------
    dict with keys: raw, master, cleaned, transformed, failed, hashed, metadata
    """
    ensure_all_dirs()
    session = requests.Session()
    session.headers.update({"User-Agent": "HDB-ETL-Pipeline/1.0"})

    # Extract
    raw_datasets = extract_all(session=session)
    master = combine_raw_datasets(raw_datasets)
    master.to_csv(RAW_DATA_FOLDER / "master_combined.csv", index=False)

    # Quality
    quality_result = run_quality_pipeline(
        master,
        reference_date=reference_date,
        flag_anomalies=flag_anomalies,
    )
    cleaned = quality_result.passed
    failed_quality = quality_result.failed

    cleaned.to_csv(CLEANED_FOLDER / "hdb_resale_cleaned.csv", index=False)
    if len(failed_quality) > 0:
        failed_quality.to_csv(FAILED_FOLDER / "failed_quality.csv", index=False)

    # Transform
    transformed, failed_transform = transform_dataset(cleaned)
    transformed.to_csv(TRANSFORMED_FOLDER / "hdb_resale_transformed.csv", index=False)

    # Combine all failed
    failed_parts = [failed_quality, failed_transform]
    all_failed = pd.concat(
        [f for f in failed_parts if len(f) > 0], ignore_index=True
    )
    if len(all_failed) > 0:
        all_failed.to_csv(FAILED_FOLDER / "hdb_resale_failed.csv", index=False)

    # Hashed
    hashed = add_hashed_column(transformed)
    hashed.to_csv(HASHED_FOLDER / "hdb_resale_hashed.csv", index=False)

    # Metadata / profiling report
    metadata = {
        "profile_report": quality_result.profile_report,
        "validation_rules": quality_result.validation_rules,
        "output_counts": {
            "API_RAW_RECORD_COUNT": len(master),
            "CLEANED_RECORD_COUNT": len(cleaned),
            "TRANSFORMED_RECORD_COUNT": len(transformed),
            "FAILED_RECORD_COUNT": len(all_failed),
            "HASHED_RECORD_COUNT": len(hashed),
        },
    }
    metadata_path = OUTPUT_FOLDER / "HDB_pipeline_metadata.json"

    def json_serializer(obj):
        if isinstance(obj, (pd.Timestamp, date)):
            return str(obj)
        if hasattr(obj, "item"):
            return obj.item()
        raise TypeError(f"Not serializable: {type(obj)}")

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, default=json_serializer)

    return {
        "raw": master,
        "cleaned": cleaned,
        "transformed": transformed,
        "failed": all_failed,
        "hashed": hashed,
        "metadata": metadata,
    }


if __name__ == "__main__":
    results = run_pipeline()
    counts = results["metadata"]["output_counts"]
    print("Pipeline completed successfully.")
    logger.info("Pipeline completed successfully.")
    for k, v in counts.items():
        print(f"  {k}: {v}")


ModuleNotFoundError: No module named 'requests'

In [78]:
from __future__ import annotations
from Source.logger import get_logger

logger = get_logger(__name__)
import json
import logging
from datetime import date
from pathlib import Path



def ensure_all_dirs() -> None:
    """Create all output subdirectories."""
    for d in [RAW_DATA_FOLDER, CLEANED_FOLDER, TRANSFORMED_FOLDER, FAILED_FOLDER, HASHED_FOLDER]:
        d.mkdir(parents=True, exist_ok=True)

logger.info("Running Pipeline Started")

def run_pipeline(
    reference_date: date | None = None,
    flag_anomalies: bool = False,
) -> dict[str, pd.DataFrame | dict]:
    """
    Execute complete ETL pipeline and write output files.

    Parameters
    ----------
    reference_date : date, optional
        Reference date for remaining lease computation (default: today).
    flag_anomalies : bool
        If True, move price anomalies to failed dataset. Default False to
        retain data while still flagging anomalies in profile report.

    Returns
    -------
    dict with keys: raw, master, cleaned, transformed, failed, hashed, metadata
    """
    ensure_all_dirs()
    session = requests.Session()
    session.headers.update({"User-Agent": "HDB-ETL-Pipeline/1.0"})

    # Extract
    raw_datasets = extract_all(session=session)
    master = combine_raw_datasets(raw_datasets)
    master.to_csv(RAW_DATA_FOLDER / "master_combined.csv", index=False)

    # Quality
    quality_result = run_quality_pipeline(
        master,
        reference_date=reference_date,
        flag_anomalies=flag_anomalies,
    )
    cleaned = quality_result.passed
    failed_quality = quality_result.failed

    cleaned.to_csv(CLEANED_FOLDER / "hdb_resale_cleaned.csv", index=False)
    if len(failed_quality) > 0:
        failed_quality.to_csv(FAILED_FOLDER / "failed_quality.csv", index=False)

    # Transform
    transformed, failed_transform = transform_dataset(cleaned)
    transformed.to_csv(TRANSFORMED_FOLDER / "hdb_resale_transformed.csv", index=False)

    # Combine all failed
    failed_parts = [failed_quality, failed_transform]
    all_failed = pd.concat(
        [f for f in failed_parts if len(f) > 0], ignore_index=True
    )
    if len(all_failed) > 0:
        all_failed.to_csv(FAILED_FOLDER / "hdb_resale_failed.csv", index=False)

    # Hashed
    hashed = add_hashed_column(transformed)
    hashed.to_csv(HASHED_FOLDER / "hdb_resale_hashed.csv", index=False)

    # Metadata / profiling report
    metadata = {
        "profile_report": quality_result.profile_report,
        "validation_rules": quality_result.validation_rules,
        "output_counts": {
            "API_RAW_RECORD_COUNT": len(master),
            "CLEANED_RECORD_COUNT": len(cleaned),
            "TRANSFORMED_RECORD_COUNT": len(transformed),
            "FAILED_RECORD_COUNT": len(all_failed),
            "HASHED_RECORD_COUNT": len(hashed),
        },
    }
    metadata_path = OUTPUT_FOLDER / "HDB_pipeline_metadata.json"

    def json_serializer(obj):
        if isinstance(obj, (pd.Timestamp, date)):
            return str(obj)
        if hasattr(obj, "item"):
            return obj.item()
        raise TypeError(f"Not serializable: {type(obj)}")

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, default=json_serializer)

    return {
        "raw": master,
        "cleaned": cleaned,
        "transformed": transformed,
        "failed": all_failed,
        "hashed": hashed,
        "metadata": metadata,
    }


if __name__ == "__main__":
    results = run_pipeline()
    counts = results["metadata"]["output_counts"]
    print("Pipeline completed successfully.")
    logger.info("Pipeline completed successfully.")
    for k, v in counts.items():
        print(f"  {k}: {v}")


2026-07-20 14:58:33 | INFO     | __main__             | Running Pipeline Started


NameError: name 'requests' is not defined